# agentic-rl-wordle — 多輪 GRPO 訓練（Colab）

流程：①參數 → ②Drive+bundle+安裝 → ③HF_TOKEN → ④pytest 煙霧 → ⑤(SMOKE專屬) spike →
⑥訓練＋（SMOKE專屬）自動續跑演練＋等待完成＋gate 判定＋（正式訓練專屬）評測+push＋釋放機器。

**這份 notebook 可以直接「全部執行」，不需要手動盯著重跑任何 cell。**

cell①有三種模式（`SMOKE_TEST` / `PILOT_MODE` 兩個旗標決定）：

| SMOKE_TEST | PILOT_MODE | 用途 | 環境 | 時長 | 評測+push |
|---|---|---|---|---|---|
| `True` | （不看） | M2.1/M2.3/M2.4：管線正確性 + 續跑演練 + 學習訊號驗證 | 猜數字（玩具） | ~40–70 分鐘（含續跑演練） | 否 |
| `False` | `True` | M3.2：Wordle 版管線短跑試驗（格式/reward hacking 肉眼檢查） | 真正 Wordle | `PILOT_HOURS`（預設 1 小時） | **否**（避免半成品模型推上 HF） |
| `False` | `False` | 正式訓練 | 真正 Wordle | `MAX_HOURS`（預設 8 小時） | 是 |

- 斷線 SOP（真的斷線，不是自動演練那種）：重新執行 ①②③⑥（RESUME="auto" 會從 Drive 最新 checkpoint 接續）
- 背景執行請開：執行階段 → 背景執行；**cell ⑥ 結尾一定會呼叫 `runtime.unassign()` 釋放機器**
  （用 try/finally 包住，就算中間任何一步出錯也會執行，不會留著空燒）

In [ ]:
# ===== ① 參數（唯一需要手改的 cell）=====
# 目前預設值已設成「正式訓練」（M3.3 重跑）——執行階段類型選 A100，「大量 RAM」要【開啟】
# （A100 的大量 RAM 開關切換的是 40GB / 80GB 兩種顯卡；40GB 實測峰值 ~44GB 會 OOM）。
SMOKE_TEST = False                # True: 猜數字煙霧；False: Wordle（正式或試跑，見下）
PILOT_MODE = False                # SMOKE_TEST=False 時生效：True=M3.2 短跑試驗（不評測不 push
                                   # HF）；False=正式訓練（跑完自動評測+push HF）
PILOT_HOURS = 1.0                 # PILOT_MODE=True 時的牆鐘預算（試跑用，跟 MAX_HOURS 分開設）
RUN_NAME = "wordle-grpo-v3"       # v1 被 6 次 OOM 崩潰寫髒；v2 訓練成功但 Drive 掛載中途無聲
                                   # 降級，權重全數遺失（見 cell⑥ 開頭的事故說明）——v3 重跑
HF_USERNAME = "steven0226"
REWARD_PRESET = "shaped"          # shaped | binary（HF 官方發現 binary 對 Wordle 更穩，留作 A/B）
MAX_HOURS = 8.0                   # 正式訓練牆鐘預算兜底（實測 ~5 秒/步，3000 步約 4-5 小時跑完）
RESUME = "auto"

DRIVE_BASE = "/content/drive/MyDrive/agentic-rl-wordle"

In [ ]:
# ===== ② Drive + 原始碼 bundle + 依賴安裝 =====
from google.colab import drive
drive.mount('/content/drive')

import pathlib
BUNDLE = f"{DRIVE_BASE}/wordle_rl_bundle.zip"
assert pathlib.Path(BUNDLE).exists(), f"先在本機執行 scripts/make_colab_bundle.py 並把 zip 上傳到 {BUNDLE}"

!rm -rf /content/agentic-rl-wordle && mkdir -p /content/agentic-rl-wordle
!unzip -q -o "$BUNDLE" -d /content/agentic-rl-wordle
%cd /content/agentic-rl-wordle

# ⚠️ 版本 pin 原則：torch 用 Colab 內建 CUDA build（絕不覆蓋，專案 2 教訓）；
#    trl/vllm/peft 精確 pin 於 requirements-colab.txt（M2.1 spike 已驗證過此組合可行）
!pip install -q -e . pytest
!pip install -q -r requirements-colab.txt

import torch, transformers, trl, vllm
print("torch", torch.__version__, "| trl", trl.__version__,
      "| transformers", transformers.__version__, "| vllm", vllm.__version__)
assert trl.__version__.startswith("1.8"), "trl 版本漂移——對照 requirements-colab.txt 與 docs/decision.md"

# 單字表 fetch-at-setup（出處與 sha256 → data/SOURCE.json；計數斷言 2315/10657/12972）
!python scripts/fetch_words.py

In [ ]:
# ===== ③ HF_TOKEN（Colab Secrets 需事先設定）=====
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN OK")

In [ ]:
# ===== ④ 煙霧驗證（<60 秒；不全綠就不要燒 GPU）=====
!python -m pytest tests -q

In [ ]:
# ===== ⑤（SMOKE 專屬）M2.1 spike：新環境首跑必執行——驗證 TRL 接觸面、記錄版本三元組 =====
# 過 → 把印出的版本回填 requirements-colab.txt、結論寫 docs/decision.md
# 不過 → 依錯誤訊息修 rollout.py / train.py / requirements-colab.txt；60 分鐘 timebox，超時轉 ART 備援
#
# ⚠️ 用 subprocess.run + assert，不用 `!python`：spike 失敗時「全部執行」必須在這裡硬停，
#    不能悶頭跑進昂貴的 cell ⑥（訓練）去空燒 GPU
#    （`!python ...` 這種 shell 魔法指令即使子行程回傳非 0，Colab 也不會讓 cell 失敗）。
# ⚠️ 一定要 capture_output=True 再自己 print 出來：子行程直接繼承 stdout 時，
#    Colab 不保證會把它顯示在 cell 輸出裡（實測只看得到後面 assert 的 traceback，
#    看不到 spike 腳本自己印的診斷內容）——用 Python 的 print() 才保證看得到。
import subprocess
import sys

if SMOKE_TEST:
    result = subprocess.run(
        [sys.executable, "scripts/spike_trl.py"], capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    assert result.returncode == 0, (
        f"M2.1 spike 失敗（returncode={result.returncode}）——修好前不要往下跑 cell ⑥"
    )

In [ ]:
# ===== ⑥ 訓練（寫本機碟）→ 週期鏡像 Drive →（SMOKE）續跑演練 → 完成後先保全權重 → gate →（正式）評測+push → 強制同步 → 釋放機器 =====
#
# ⚠️ v2 事故說明（M3.3 第一次成功訓練的權重全數遺失，本 cell 因此改版）：
#    Colab 的 Drive 掛載在長時間高頻寫入（checkpoint 輪替 24 次 + 每 50 步 samples +
#    每步 append metrics/log）下會「無聲降級」——實測 5:27pm 後所有「新建檔案」都沒有
#    真正落地（checkpoint 目錄變空殼、samples 停在 step 2450、final/ 完全消失），但
#    「既有檔案」的 append 照常成功，訓練行程零錯誤跑完 rc=0，事後才發現權重不存在。
#    對策（本版）：
#    1. 訓練一律寫【本機碟】/content/runs/...（快且可靠），不直接寫 Drive
#    2. cell⑥ 每 15 分鐘把最新 checkpoint + metrics + log + 新 samples「低頻單向」鏡像到
#       Drive（斷線後重跑 ①②③⑥ 會自動從 Drive 備份還原再 --resume auto 接續）
#    3. 訓練成功後【第一件事】把 final/ 複製到 Drive 並逐檔驗證大小——權重先保全，
#       之後評測/push 再出任何問題都不會再毀掉一次訓練（救援 notebook 可接手）
#    4. 結束前 drive.flush_and_unmount() 強制同步，再 runtime.unassign()
#
# 崩潰自動重啟（首次 FULL 的 OOM 事故加入）與 SMOKE 的 M2.4 續跑演練照舊保留。

import json
import pathlib
import shutil
import subprocess
import sys
import time

_suffix = "-smoke" if SMOKE_TEST else ("-pilot" if PILOT_MODE else "")
LOCAL_DIR = pathlib.Path(f"/content/runs/{RUN_NAME}{_suffix}")      # 訓練實際寫這裡
DRIVE_DIR = pathlib.Path(f"{DRIVE_BASE}/runs/{RUN_NAME}{_suffix}")  # 備份與交付
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOCAL_DIR / "train.log"
METRICS_PATH = LOCAL_DIR / "metrics.jsonl"

MAX_CRASH_RETRIES = 5      # 崩潰自動重啟上限
MIRROR_EVERY_SEC = 15 * 60  # Drive 鏡像頻率（低頻，避免觸發 v2 的高頻寫入降級）


def _budgeted_hours():
    if SMOKE_TEST:
        return 0.5
    return PILOT_HOURS if PILOT_MODE else MAX_HOURS


def newest_ckpt(root):
    """回傳 (step, path)：root 底下 trainer_state.json 完好的最大 step checkpoint。"""
    best = None
    for p in pathlib.Path(root).glob("checkpoint-*"):
        try:
            step = int(p.name.split("-")[-1])
        except ValueError:
            continue
        if (p / "trainer_state.json").exists() and (best is None or step > best[0]):
            best = (step, p)
    return best


_last_mirrored_step = -1


def mirror_to_drive():
    """低頻單向備份：metrics/log 覆蓋、samples 補新檔、最新 checkpoint 只在 Drive 留一份。
    備份失敗只警告不打斷訓練（訓練的真相在本機碟）。"""
    global _last_mirrored_step
    try:
        for name in ("metrics.jsonl", "train.log"):
            src = LOCAL_DIR / name
            if src.exists():
                shutil.copy2(src, DRIVE_DIR / name)
        s_src, s_dst = LOCAL_DIR / "samples", DRIVE_DIR / "samples"
        if s_src.exists():
            s_dst.mkdir(exist_ok=True)
            for f in s_src.glob("*.md"):
                if not (s_dst / f.name).exists():
                    shutil.copy2(f, s_dst / f.name)
        b = newest_ckpt(LOCAL_DIR)
        if b is not None and b[0] != _last_mirrored_step:
            tmp = DRIVE_DIR / (b[1].name + ".tmp")
            shutil.rmtree(tmp, ignore_errors=True)
            shutil.copytree(b[1], tmp)
            dst = DRIVE_DIR / b[1].name
            shutil.rmtree(dst, ignore_errors=True)
            tmp.rename(dst)
            for old in DRIVE_DIR.glob("checkpoint-*"):
                if old.name != b[1].name:
                    shutil.rmtree(old, ignore_errors=True)
            _last_mirrored_step = b[0]
            print(f"[mirror] checkpoint-{b[0]} → Drive ✓", flush=True)
    except Exception as e:
        print(f"[mirror] 備份失敗（不影響訓練，下輪再試）：{e}", flush=True)


def launch_training(hours=None):
    preset = "smoke" if SMOKE_TEST else "full"
    hours = _budgeted_hours() if hours is None else hours
    cmd = [sys.executable, "-m", "wordle_rl.train",
           "--preset", preset,
           "--reward", REWARD_PRESET,
           "--output-dir", str(LOCAL_DIR),
           "--resume", RESUME,
           "--max-hours", str(hours)]
    print(">>>", " ".join(cmd), flush=True)
    log_f = open(LOG_PATH, "ab")
    p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)
    print("PID:", p.pid, "| log:", LOG_PATH, flush=True)
    return p


def run_logged(cmd, log_name):
    """評測/push 專用。⚠️ 子行程直接繼承 stdout 時 Colab 常常什麼都不顯示（cell⑤ 同款
    教訓）——寫本機 log 檔再 print 尾端，並順手備份一份到 Drive。"""
    log_path = LOCAL_DIR / log_name
    print(">>>", " ".join(map(str, cmd)), "| log:", log_path, flush=True)
    with open(log_path, "ab") as f:
        rc = subprocess.call([str(c) for c in cmd], stdout=f, stderr=subprocess.STDOUT)
    print("\n".join(log_path.read_text(errors="ignore").splitlines()[-40:]), flush=True)
    try:
        shutil.copy2(log_path, DRIVE_DIR / log_name)
    except Exception:
        pass
    assert rc == 0, f"失敗（returncode={rc}），完整 log：{log_path}"


def read_metrics():
    if not METRICS_PATH.exists():
        return []
    return [json.loads(l) for l in METRICS_PATH.read_text().splitlines() if l.strip()]


def tail_log(n=30):
    if not LOG_PATH.exists():
        return "(尚無 log)"
    return "\n".join(LOG_PATH.read_text(errors="ignore").splitlines()[-n:])


# ---------- 斷線復原：本機沒有 checkpoint 但 Drive 備份有 → 先還原再開跑 ----------
if newest_ckpt(LOCAL_DIR) is None:
    b = newest_ckpt(DRIVE_DIR)
    if b is not None:
        print(f"從 Drive 備份還原 {b[1].name} 到本機（斷線復原）…", flush=True)
        shutil.copytree(b[1], LOCAL_DIR / b[1].name)
        for name in ("metrics.jsonl", "train.log"):
            src = DRIVE_DIR / name
            if src.exists() and not (LOCAL_DIR / name).exists():
                shutil.copy2(src, LOCAL_DIR / name)

resume_drill_result = "SKIPPED（FULL/PILOT 模式不做 M2.4 演練）"
crash_retry_count = 0
training_ok = False

try:
    proc = launch_training()

    # ---------- M2.4 續跑演練（只在 SMOKE_TEST 做）----------
    if SMOKE_TEST:
        print("\n=== M2.4 續跑演練：等待第一個 checkpoint 出現（最長 20 分鐘）===", flush=True)
        deadline = time.monotonic() + 20 * 60
        ckpt = None
        while time.monotonic() < deadline:
            ckpt = newest_ckpt(LOCAL_DIR)
            if ckpt is not None:
                break
            time.sleep(20)

        if ckpt is None:
            resume_drill_result = "FAIL（20 分鐘內沒出現 checkpoint，train.log 尾端如下）"
            print("❌", resume_drill_result, "\n", tail_log(), flush=True)
        else:
            ckpt_step = ckpt[0]
            n_before = len(read_metrics())
            print(f"✓ checkpoint 出現，global_step={ckpt_step}；SIGKILL 模擬斷線 …", flush=True)
            proc.kill()
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print("⚠ 行程 30 秒內未結束（繼續往下走）", flush=True)
            time.sleep(5)
            print("以 --resume auto 重新啟動 …", flush=True)
            proc = launch_training()

            print("等待重啟後寫入新的 metrics 紀錄（最長 10 分鐘）…", flush=True)
            deadline = time.monotonic() + 10 * 60
            resumed_first_step = None
            while time.monotonic() < deadline:
                rows = read_metrics()
                if len(rows) > n_before:
                    resumed_first_step = rows[n_before]["step"]
                    break
                time.sleep(15)

            if resumed_first_step is None:
                resume_drill_result = "FAIL（重啟後 10 分鐘內沒有新的 metrics 紀錄）"
            elif resumed_first_step <= 2 and ckpt_step > 5:
                resume_drill_result = (
                    f"FAIL（重啟後第一步 step={resumed_first_step}，"
                    f"疑似從頭重跑而非接續 checkpoint step={ckpt_step}）"
                )
            else:
                resume_drill_result = (
                    f"PASS（checkpoint step={ckpt_step} → 重啟後接續 step={resumed_first_step}）"
                )
            print(("✅ " if resume_drill_result.startswith("PASS") else "❌ ") + resume_drill_result,
                  flush=True)

    # ---------- 等待訓練結束：每 30 秒 poll、每 15 分鐘鏡像 Drive、非 0 結束自動重啟 ----------
    budget_deadline = time.monotonic() + _budgeted_hours() * 3600
    print(f"\n=== 等待訓練程序結束（總牆鐘預算 {_budgeted_hours()} 小時；"
          f"每 {MIRROR_EVERY_SEC // 60} 分鐘鏡像 Drive；"
          f"非正常結束自動 --resume auto 重啟，最多 {MAX_CRASH_RETRIES} 次）===", flush=True)
    while True:
        next_mirror = time.monotonic() + MIRROR_EVERY_SEC
        while True:
            rc = proc.poll()
            if rc is not None:
                break
            if time.monotonic() >= next_mirror:
                mirror_to_drive()
                next_mirror = time.monotonic() + MIRROR_EVERY_SEC
            time.sleep(30)
        print(f"訓練程序結束 returncode = {rc}", flush=True)
        if rc == 0:
            training_ok = True
            break
        print(f"\n❌ 訓練行程非正常結束（returncode={rc}），log 尾端如下：\n{tail_log(60)}", flush=True)
        mirror_to_drive()  # 崩潰當下先保全一次進度
        remaining_hours = (budget_deadline - time.monotonic()) / 3600
        if crash_retry_count >= MAX_CRASH_RETRIES:
            print(f"❌ 已達最大自動重啟次數（{MAX_CRASH_RETRIES}），放棄自動重啟——"
                  "人工檢視上面的 log", flush=True)
            break
        if remaining_hours <= 0.05:
            print("❌ 牆鐘預算已用完，不再重啟", flush=True)
            break
        crash_retry_count += 1
        print(f"⏳ 剩餘預算 {remaining_hours:.2f} 小時，5 秒後以 --resume auto 重啟"
              f"（第 {crash_retry_count}/{MAX_CRASH_RETRIES} 次自動重啟）…", flush=True)
        time.sleep(5)
        proc = launch_training(hours=remaining_hours)

    # ---------- 訓練成功 → 第一件事：保全權重到 Drive（逐檔驗證大小）----------
    adapter_local = None
    if training_ok:
        adapter_local = LOCAL_DIR / "final"
        if not (adapter_local / "adapter_config.json").exists():
            b = newest_ckpt(LOCAL_DIR)
            assert b is not None and (b[1] / "adapter_config.json").exists(), \
                "本機找不到 final/ 也沒有含 adapter 的 checkpoint——檢視 train.log"
            adapter_local = b[1]
            print(f"⚠ final/ 不存在，改用 {adapter_local.name} 作為交付 adapter", flush=True)
        final_drive = DRIVE_DIR / "final"
        shutil.rmtree(final_drive, ignore_errors=True)
        shutil.copytree(adapter_local, final_drive)
        for f in adapter_local.rglob("*"):
            if f.is_file():
                g = final_drive / f.relative_to(adapter_local)
                assert g.exists() and g.stat().st_size == f.stat().st_size, \
                    f"Drive 權重備份驗證失敗：{g}（大小不符或不存在）"
        print(f"✅ 權重已保全到 {final_drive}（逐檔大小驗證通過）", flush=True)
        mirror_to_drive()

    # ---------- M2.3 gate 自動判定（reward/win_rate 是否有上升）----------
    rows = read_metrics()
    _label = "SMOKE" if SMOKE_TEST else ("PILOT" if PILOT_MODE else "FULL")
    print(f"\n=== gate（{_label}，共 {len(rows)} 筆 metrics 紀錄）===", flush=True)
    if len(rows) >= 15:
        early = rows[:10]
        late = rows[-20:] if len(rows) >= 20 else rows[-max(1, len(rows) // 3):]

        def mean(key, subset):
            vals = [r[key] for r in subset if r.get(key) is not None]
            return sum(vals) / len(vals) if vals else None

        early_r, late_r = mean("reward/mean", early), mean("reward/mean", late)
        early_wr, late_wr = mean("rollout/win_rate", early), mean("rollout/win_rate", late)
        print(f"reward/mean   前段={early_r}  後段={late_r}")
        print(f"win_rate      前段={early_wr}  後段={late_wr}")
        gate_pass = (
            late_r is not None and early_r is not None and late_r > early_r
            and late_wr is not None and early_wr is not None and (late_wr - early_wr) >= 0.15
        )
        # ⚠️ 15pp win_rate 門檻是照玩具環境（猜數字）校準的，對 Wordle 偏嚴——
        #    正式成敗由 M3.4 評測（200 保留詞、對未訓練 baseline、Wilson CI）判定，這裡僅供參考。
        print("✅ GATE PASS" if gate_pass
              else "❌ GATE 未過（reward 或 win_rate 沒有明顯上升，人工檢視 samples/ 判斷原因）")
    else:
        print("樣本數不足以自動判定，人工檢視 metrics.jsonl / train.log")

    # ---------- 曲線圖（本機存一份、Drive 存一份）----------
    if rows:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
        for ax, key, title in [
            (axes[0], "reward/mean", "mean episode reward"),
            (axes[1], "rollout/win_rate", "rollout win rate"),
            (axes[2], "rollout/illegal_per_ep", "illegal turns / episode"),
        ]:
            pts = [(r["step"], r[key]) for r in rows if r.get(key) is not None]
            if pts:
                ax.plot(*zip(*pts))
            ax.set_title(title)
            ax.set_xlabel("step")
        plt.tight_layout()
        plt.savefig(LOCAL_DIR / "curves.png")
        try:
            shutil.copy2(LOCAL_DIR / "curves.png", DRIVE_DIR / "curves.png")
        except Exception:
            pass
        plt.show()

    # ---------- 正式訓練專屬：評測 + push HF（權重已保全，這裡再出錯也不會毀掉訓練）----------
    if training_ok and not SMOKE_TEST and not PILOT_MODE:
        run_logged([sys.executable, "eval/run_eval.py",
                    "--adapter", str(adapter_local), "--backend", "vllm"], "eval.log")
        shutil.copytree("results", DRIVE_DIR / "results", dirs_exist_ok=True)
        report = pathlib.Path("results/final_report.md")
        if report.exists():
            print("\n" + "=" * 28 + " final_report.md " + "=" * 28 + "\n", flush=True)
            print(report.read_text(encoding="utf-8"), flush=True)
        run_logged([sys.executable, "scripts/push_model.py",
                    "--adapter", str(adapter_local),
                    "--repo", f"{HF_USERNAME}/qwen2.5-1.5b-wordle-grpo",
                    "--card", "docs/model_card.md"], "push.log")
        print("HF push 完成 ✅", flush=True)
    elif training_ok and PILOT_MODE:
        print(f"\nPILOT_MODE：不評測不 push；人工檢視 {DRIVE_DIR}/samples/ 的 transcript 判斷 "
              "Wordle 版管線是否正常（格式錯誤率、有無 reward hacking）。", flush=True)

    print(f"\n=== 總結 ===\n訓練成功：{training_ok}\nM2.4 續跑演練：{resume_drill_result}\n"
          f"崩潰自動重啟：共 {crash_retry_count} 次（上限 {MAX_CRASH_RETRIES}）", flush=True)
finally:
    try:
        mirror_to_drive()
    except Exception:
        pass
    try:
        from google.colab import drive as _gdrive
        _gdrive.flush_and_unmount()
        print("Drive 已強制同步（flush_and_unmount）✓", flush=True)
    except Exception as e:
        print(f"⚠ flush_and_unmount 失敗：{e}", flush=True)
    from google.colab import runtime
    runtime.unassign()